# VGG-Style CNN on CIFAR-10: An Ablation Study of Augmentation, BatchNorm and Dropout

**Day 2 of 10 · Deep Learning Architectures Series**

In this notebook I build a **VGG-style convolutional network** from scratch in PyTorch and train it on CIFAR-10. Instead of only reporting one accuracy number, I run a controlled **ablation study**: starting from a plain baseline that overfits heavily, I add data augmentation, Batch Normalization and Dropout one at a time to measure what each technique actually contributes.

**What's inside**
1. Background: from AlexNet to VGG
2. Data pipeline with a proper train/validation/test split
3. Model implementation and parameter verification
4. Five controlled experiments (A to E)
5. Results: comparison table, overfitting analysis, confusion matrix, per-class accuracy, errors
6. Learned first-layer filters
7. Analysis, limitations, next steps

> Papers: *ImageNet Classification with Deep CNNs* (Krizhevsky et al., 2012) · *Very Deep Convolutional Networks for Large-Scale Image Recognition* (Simonyan and Zisserman, 2014)

> ⚙️ **Kaggle setup:** Settings → Accelerator → **GPU (T4 / P100)** and **Internet → On** (needed to download CIFAR-10). Expected runtime: roughly 15 to 30 minutes for all five experiments.

## 1. Background: from AlexNet to VGG

**AlexNet (2012)** won ImageNet by a large margin and made deep learning mainstream. Its ingredients: ReLU activations, Dropout in the fully connected layers, data augmentation, and GPU training. Its first layer uses large 11×11 filters with stride 4, which is designed for 224×224 ImageNet images and is a poor fit for 32×32 CIFAR images.

**VGG (2014)** simplified the design with one rule: *use only 3×3 convolutions and stack many of them*, with 2×2 max-pooling between blocks while doubling the channels (64 → 128 → 256 → ...).

**Why stacked 3×3 convolutions?** A stack of small filters has the same receptive field as one large filter, with fewer parameters and more non-linearities. For $C$ input and output channels:

| Option | Receptive field | Parameters |
|--------|----------------|-----------|
| one 5×5 conv | 5×5 | $25C^2$ |
| two 3×3 convs | 5×5 | $18C^2$ |
| one 7×7 conv | 7×7 | $49C^2$ |
| three 3×3 convs | 7×7 | $27C^2$ |

From a signal-processing view, cascading two FIR filters convolves their kernels, so the effective support grows: a 3×3 kernel convolved with another 3×3 kernel has 5×5 support.

**Adaptation for CIFAR-10:** inputs are 32×32, so I use 3 convolutional blocks (6 conv layers, called *VGG-6* here) followed by a small classifier. The `CFGS` dictionary also contains a full VGG-11 configuration you can try as an extension.

### The regularization toolbox
- **Data augmentation:** random crop (4-pixel padding) and horizontal flip. It injects invariances the network would otherwise have to learn from limited data.
- **Batch Normalization:** normalizes each channel using batch statistics, then learns a scale and shift. It stabilizes training, allows faster learning, and has a mild regularizing effect.
- **Dropout:** randomly zeroes units with probability $p$ during training, preventing co-adaptation (similar to training an ensemble of sub-networks).

### Experiment design

| ID | Augmentation | BatchNorm | Dropout |
|----|:-----------:|:---------:|:-------:|
| A: baseline | – | – | – |
| B: + augmentation | ✓ | – | – |
| C: + batchnorm | – | ✓ | – |
| D: augmentation + batchnorm | ✓ | ✓ | – |
| E: augmentation + batchnorm + dropout | ✓ | ✓ | 0.5 |

Everything else (architecture, optimizer, learning-rate schedule, seed, epochs) is identical, so differences come from the technique under test. No weight decay is used, to isolate these three effects.

In [ ]:
import copy, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
VAL_SIZE = 5000
ARCH = "vgg6"          # try "vgg11" as an extension

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)

def save_fig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")

print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 2. Data Pipeline

CIFAR-10 has 50,000 training and 10,000 test images (32×32 RGB, 10 classes). I split the training set into **45,000 train + 5,000 validation**. The validation set is used to pick the best epoch of each run, so the **test set is touched only once per experiment** and is not used for model selection.

Augmentation is applied **only to training data**. Validation and test images are just normalized.

In [ ]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
normalize = transforms.Normalize(mean, std)

tf_plain = transforms.Compose([transforms.ToTensor(), normalize])
tf_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

full_plain = datasets.CIFAR10("./data", train=True, download=True, transform=tf_plain)
full_aug   = datasets.CIFAR10("./data", train=True, download=False, transform=tf_aug)
test_ds    = datasets.CIFAR10("./data", train=False, download=True, transform=tf_plain)
classes = full_plain.classes

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(full_plain), generator=g).tolist()
val_idx, train_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]

train_plain = Subset(full_plain, train_idx)   # no augmentation
train_aug   = Subset(full_aug,   train_idx)   # with augmentation
val_ds      = Subset(full_plain, val_idx)

def make_loader(ds, shuffle=False, bs=BATCH_SIZE):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

val_dl, test_dl = make_loader(val_ds, bs=512), make_loader(test_ds, bs=512)
train_eval_dl = make_loader(train_plain, bs=512)   # clean pass over train data for the final gap

print(f"train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_ds):,}")
print("classes:", classes)

In [ ]:
mean_t = torch.tensor(mean).view(3, 1, 1)
std_t  = torch.tensor(std).view(3, 1, 1)
def denorm(x):
    return (x * std_t + mean_t).clamp(0, 1)

# Samples + what augmentation does to a single image
raw = datasets.CIFAR10("./data", train=True, download=False)
aug_demo = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])

fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for i in range(8):
    img, lab = raw[i]
    axes[0, i].imshow(img); axes[0, i].set_title(classes[lab], fontsize=9); axes[0, i].axis("off")
img, lab = raw[0]
axes[1, 0].imshow(img); axes[1, 0].set_title("original", fontsize=9); axes[1, 0].axis("off")
for i in range(1, 8):
    axes[1, i].imshow(aug_demo(img)); axes[1, i].set_title("augmented", fontsize=9); axes[1, i].axis("off")
plt.suptitle("Top: CIFAR-10 samples | Bottom: random augmentations of one image (crop + flip)")
plt.tight_layout()
save_fig("data_and_augmentation.png")
plt.show()

## 3. Model Implementation

One class covers all five experiments through three switches: `batch_norm`, `dropout`, and the architecture name.

Details:
- Each conv block: `Conv3x3 → (BatchNorm) → ReLU`, with `padding=1` so spatial size is preserved; `MaxPool2d(2)` halves it between blocks.
- When BatchNorm follows a conv, the conv bias is redundant, so it is dropped (`bias=not batch_norm`).
- Dropout is applied in the classifier only, as in AlexNet and VGG.
- Weights use Kaiming (He) initialization, which suits ReLU networks and lets the no-BatchNorm baseline train reliably.

| Layer | Output shape | Parameters (no BN) |
|-------|--------------|-------------------|
| Input | 3×32×32 | 0 |
| Conv 3→64, Conv 64→64 | 64×32×32 | 1,792 + 36,928 |
| MaxPool | 64×16×16 | 0 |
| Conv 64→128, Conv 128→128 | 128×16×16 | 73,856 + 147,584 |
| MaxPool | 128×8×8 | 0 |
| Conv 128→256, Conv 256→256 | 256×8×8 | 295,168 + 590,080 |
| MaxPool | 256×4×4 | 0 |
| FC 4096→512 | 512 | 2,097,664 |
| FC 512→10 | 10 | 5,130 |
| **Total** | | **3,248,202** |

In [ ]:
CFGS = {
    "vgg6":  [64, 64, "M", 128, 128, "M", 256, 256, "M"],
    "vgg11": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
}

class VGGNet(nn.Module):
    def __init__(self, arch="vgg6", batch_norm=False, dropout=0.0, num_classes=10):
        super().__init__()
        layers, in_ch, n_pool = [], 3, 0
        for v in CFGS[arch]:
            if v == "M":
                layers.append(nn.MaxPool2d(2, 2)); n_pool += 1
            else:
                layers.append(nn.Conv2d(in_ch, v, kernel_size=3, padding=1, bias=not batch_norm))
                if batch_norm:
                    layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_ch = v
        self.features = nn.Sequential(*layers)
        spatial = 32 // (2 ** n_pool)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * spatial * spatial, 512), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters())

m = VGGNet(ARCH)
print(m)
print(f"\nParameters (no BN, no dropout): {count_params(m):,}")
if ARCH == "vgg6":
    assert count_params(m) == 3_248_202, "unexpected parameter count"
print(f"Parameters (with BN): {count_params(VGGNet(ARCH, batch_norm=True)):,}")

In [ ]:
# Shape trace
x = torch.randn(1, 3, 32, 32)
for layer in m.features:
    x = layer(x)
    if isinstance(layer, (nn.MaxPool2d,)):
        print(f"{layer.__class__.__name__:<10} -> {tuple(x.shape)}")
print("classifier ->", tuple(m.classifier(x).shape))

## 4. Training Setup

- **Loss:** Cross-Entropy
- **Optimizer:** Adam, lr = 1e-3, no weight decay
- **Schedule:** cosine annealing over 20 epochs
- **Mixed precision** (AMP) on GPU for speed
- **Model selection:** the epoch with the best **validation** accuracy; the test set is evaluated once with those weights
- **Fair comparison:** same seed and same hyperparameters for every run

For each run I also measure accuracy on the **clean (non-augmented) training set in eval mode**. This gives an honest generalization gap (train − test), because the accuracy logged during training is computed on augmented data with dropout active.

In [ ]:
use_amp = DEVICE.type == "cuda"

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            n          += x.size(0)
    return total_loss / n, correct / n


def train_experiment(name, augment, batch_norm, dropout):
    print(f"\n=== {name} | aug={augment} bn={batch_norm} dropout={dropout} ===")
    torch.manual_seed(SEED)
    model = VGGNet(ARCH, batch_norm=batch_norm, dropout=dropout).to(DEVICE)
    train_dl = make_loader(train_aug if augment else train_plain, shuffle=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val, best_epoch, best_state = -1.0, 0, None
    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_dl, criterion, optimizer, scaler)
        va_loss, va_acc = run_epoch(model, val_dl, criterion)
        scheduler.step()
        for k, v in zip(hist, [tr_loss, va_loss, tr_acc, va_acc]):
            hist[k].append(v)
        if va_acc > best_val:
            best_val, best_epoch = va_acc, ep
            best_state = copy.deepcopy(model.state_dict())
        print(f"Epoch {ep:02d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.4f} "
              f"| val loss {va_loss:.3f} acc {va_acc:.4f}")

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    _, test_acc = run_epoch(model, test_dl, criterion)
    _, train_acc_clean = run_epoch(model, train_eval_dl, criterion)
    print(f"-> best epoch {best_epoch} | val {best_val:.4f} | test {test_acc:.4f} "
          f"| clean train {train_acc_clean:.4f} | {elapsed:.0f}s")
    return dict(model=model, hist=hist, best_epoch=best_epoch, best_val_acc=best_val,
                test_acc=test_acc, train_acc_clean=train_acc_clean,
                params=count_params(model), time=elapsed)

results = {}

### Experiment A: baseline

In [ ]:
results["A: baseline"] = train_experiment("A: baseline", augment=False, batch_norm=False, dropout=0.0)

### Experiment B: +augmentation

In [ ]:
results["B: +augmentation"] = train_experiment("B: +augmentation", augment=True, batch_norm=False, dropout=0.0)

### Experiment C: +batchnorm

In [ ]:
results["C: +batchnorm"] = train_experiment("C: +batchnorm", augment=False, batch_norm=True, dropout=0.0)

### Experiment D: aug+batchnorm

In [ ]:
results["D: aug+batchnorm"] = train_experiment("D: aug+batchnorm", augment=True, batch_norm=True, dropout=0.0)

### Experiment E: aug+bn+dropout

In [ ]:
results["E: aug+bn+dropout"] = train_experiment("E: aug+bn+dropout", augment=True, batch_norm=True, dropout=0.5)

## 5. Results

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Config": name,
        "Params": f"{r['params']:,}",
        "Best epoch": r["best_epoch"],
        "Val acc (%)": round(r["best_val_acc"] * 100, 2),
        "Test acc (%)": round(r["test_acc"] * 100, 2),
        "Train acc, clean (%)": round(r["train_acc_clean"] * 100, 2),
        "Gap train-test (pp)": round((r["train_acc_clean"] - r["test_acc"]) * 100, 2),
        "Time (s)": round(r["time"]),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "results_summary.csv", index=False)
summary

In [ ]:
# Training curves for every experiment
names = list(results)
ep = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(2, len(names), figsize=(4.2 * len(names), 7))
for j, name in enumerate(names):
    h = results[name]["hist"]
    axes[0, j].plot(ep, np.array(h["train_acc"]) * 100, "--", label="train")
    axes[0, j].plot(ep, np.array(h["val_acc"]) * 100, label="val")
    axes[0, j].set_title(name); axes[0, j].set_ylim(30, 101); axes[0, j].set_xlabel("epoch")
    axes[1, j].plot(ep, h["train_loss"], "--", label="train")
    axes[1, j].plot(ep, h["val_loss"], label="val")
    axes[1, j].set_xlabel("epoch")
    if j == 0:
        axes[0, j].set_ylabel("accuracy (%)"); axes[1, j].set_ylabel("loss")
        axes[0, j].legend(); axes[1, j].legend()
plt.suptitle("Training curves (train accuracy is measured on augmented data / with dropout when those are on)")
plt.tight_layout()
save_fig("training_curves_all.png")
plt.show()

In [ ]:
# Overfitting analysis + head-to-head comparison
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
for name in names:
    h = results[name]["hist"]
    ax[0].plot(ep, np.array(h["val_acc"]) * 100, label=name)
    gap_curve = np.array(h["val_loss"]) - np.array(h["train_loss"])
    ax[1].plot(ep, gap_curve, label=name)
ax[0].set_title("Validation accuracy (%)"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].axhline(0, color="gray", lw=0.8, ls="--")
ax[1].set_title("Loss gap per epoch (val - train)"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=0.3)

gaps = [(r["train_acc_clean"] - r["test_acc"]) * 100 for r in results.values()]
bars = ax[2].bar(range(len(names)), gaps, color="tab:red", alpha=0.75)
ax[2].set_xticks(range(len(names))); ax[2].set_xticklabels([n.split(":")[0] for n in names])
ax[2].set_title("Final generalization gap: clean train - test (pp)")
for b, g_ in zip(bars, gaps):
    ax[2].text(b.get_x() + b.get_width() / 2, g_ + 0.2, f"{g_:.1f}", ha="center", fontsize=9)
plt.tight_layout()
save_fig("ablation_summary.png")
plt.show()

In [ ]:
# Select the best configuration by VALIDATION accuracy (not test) and analyze it
best_name = max(results, key=lambda k: results[k]["best_val_acc"])
best_model = results[best_name]["model"]
print("Best config by validation accuracy:", best_name,
      f"| test acc {results[best_name]['test_acc']*100:.2f}%")

@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, L, X = [], [], []
    for x, y in loader:
        P.append(model(x.to(DEVICE)).argmax(1).cpu()); L.append(y); X.append(x)
    return torch.cat(P).numpy(), torch.cat(L).numpy(), torch.cat(X)

preds, labels, imgs = predict(best_model, test_dl)
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=classes, yticklabels=classes)
ax[0].set_title(f"Confusion matrix - {best_name}")
cm_err = cm.copy(); np.fill_diagonal(cm_err, 0)
sns.heatmap(cm_err, annot=True, fmt="d", cmap="Reds", cbar=False, ax=ax[1],
            xticklabels=classes, yticklabels=classes)
ax[1].set_title("Errors only (diagonal removed)")
for a in ax:
    a.set_xlabel("Predicted"); a.set_ylabel("True")
plt.tight_layout()
save_fig("confusion_matrix.png")
plt.show()

print(classification_report(labels, preds, target_names=classes, digits=3))

In [ ]:
# Per-class accuracy + most confused pairs
per_class = cm.diagonal() / cm.sum(1)
order = np.argsort(per_class)
plt.figure(figsize=(8, 4))
plt.barh([classes[i] for i in order], per_class[order] * 100, color="tab:blue")
plt.xlabel("accuracy (%)"); plt.title(f"Per-class test accuracy - {best_name}")
plt.xlim(max(0, per_class.min() * 100 - 10), 100)
plt.tight_layout()
save_fig("per_class_accuracy.png")
plt.show()

top = np.dstack(np.unravel_index(np.argsort(cm_err.ravel())[::-1][:5], cm.shape))[0]
print("Top-5 confusions (true -> predicted : count)")
for t, p in top:
    print(f"  {classes[t]} -> {classes[p]} : {cm_err[t, p]}")

In [ ]:
# Misclassified test images
wrong = np.where(preds != labels)[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
for a, i in zip(axes.ravel(), wrong):
    a.imshow(denorm(imgs[i]).permute(1, 2, 0))
    a.set_title(f"T: {classes[labels[i]]}\nP: {classes[preds[i]]}", fontsize=8, color="red")
    a.axis("off")
plt.suptitle("Misclassified test images (T = true, P = predicted)")
plt.tight_layout()
save_fig("misclassified.png")
plt.show()

## 6. What Did the First Layer Learn?

The first convolution has 64 filters of size 3×3×3 (RGB). Each one is shown below as a tiny color image, min-max normalized for display. Early filters typically become oriented edge detectors and color-opponent blobs.

In [ ]:
w = best_model.features[0].weight.detach().cpu()        # (64, 3, 3, 3)
w = (w - w.amin(dim=(1, 2, 3), keepdim=True)) / (w.amax(dim=(1, 2, 3), keepdim=True) - w.amin(dim=(1, 2, 3), keepdim=True) + 1e-8)

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for a, f in zip(axes.ravel(), w):
    a.imshow(f.permute(1, 2, 0)); a.axis("off")
plt.suptitle(f"Learned first-layer filters (3x3 RGB) - {best_name}")
plt.tight_layout()
save_fig("conv1_filters.png")
plt.show()

In [ ]:
# Save weights of every experiment
for name, r in results.items():
    fname = "vgg_" + name.split(":")[0].lower() + ".pth"
    torch.save(r["model"].state_dict(), OUT / fname)
print("Saved to", OUT)

## 7. Analysis

### Key findings

**1. The baseline overfits severely.** Model A reaches 100% clean training accuracy but only 78.79% test accuracy, a gap of 21.2 percentage points (pp). Its validation loss bottoms out around epoch 5 (about 0.7) and then climbs to about 1.8, while validation accuracy keeps creeping up to about 80%. In other words, the network becomes increasingly overconfident on its mistakes, so accuracy alone hides the overfitting and the validation loss curve is essential.

**2. Every technique helped.** Test accuracy gains over the baseline: augmentation +6.9 pp (B), BatchNorm +7.2 pp (C), both together +10.1 pp (D), and both plus dropout +7.9 pp (E).

**3. Augmentation and BatchNorm solve different problems.**
- *BatchNorm* mainly improves optimization: validation accuracy is higher from the early epochs and the validation loss stays flat near 0.6 instead of exploding. But the network still memorizes the training set (100% clean train accuracy), so a 14.0 pp gap remains.
- *Augmentation* prevents memorization: clean train accuracy is only 93.6%, validation loss is still decreasing at epoch 20, and the gap drops to 7.9 pp.
- Between the two, BatchNorm is slightly ahead on test accuracy (85.98% vs 85.67%), but a 0.3 pp difference is within noise (see limitations), so I do not rank them.

**4. The gains are sub-additive.** B and C individually add 6.9 and 7.2 pp, which would sum to 14.1 pp, but combined (D) they add 10.1 pp. The two techniques partly overlap, so each one gives less once the other is present.

**5. Dropout hurt in this setup.** Adding dropout (p = 0.5) to the best configuration lowered test accuracy from 88.93% to 86.70% (−2.2 pp), even though E has the smallest generalization gap (4.3 pp). Its clean training accuracy is only 91.0%, and in the curves the validation accuracy sits *above* the training accuracy (the training numbers are measured with augmentation and dropout active). This is an underfitting regime: a small gap is not the goal, a low test error is. A plausible explanation is that p = 0.5 plus augmentation is too much regularization for a small network and a 20-epoch budget. I have not tested a lower rate (0.2 to 0.3) or a longer schedule, so this remains a hypothesis.

**6. The model is probably undertrained.** The best epoch of both B and D is the last one (epoch 20), and their validation loss is still decreasing. A longer schedule would likely improve the augmented runs further.

**7. Error analysis (best model, D).**
- *Hardest classes:* cat (77.2% recall), dog (82.5%), bird (84.0%). *Easiest:* automobile (95.3%), truck (94.4%), ship (94.4%).
- The top two confusions are **dog → cat (102)** and **cat → dog (88)**: these 190 errors make up about 17% of all 1,107 test errors. Other frequent confusions: bird → cat (39), cat → deer (39), airplane → ship (39), and automobile ↔ truck (26 each way).
- Errors concentrate inside groups of visually similar classes (four-legged animals, vehicles). A plausible reason is the 32×32 resolution: fine details such as ears and snouts are mostly lost, while pose, fur texture and background vary a lot. Airplane/ship confusions may come from similar sky and water backgrounds (also a hypothesis).
- The misclassified samples are a mix of genuinely hard cases (a toy horse figurine predicted as truck, a close-up of a bird's head predicted as cat, unusual crops) and clear model failures (a dog standing on grass predicted as deer).

**8. Cost.** Augmentation made runs about 20 to 30% slower (225 to 227 s vs 169 to 191 s), likely because cropping and flipping run on the CPU data loader.

### Limitations
- Single run per configuration (one seed). The difference between validation and test accuracy is 0.3 to 1.5 pp across runs, which gives a rough sense of the noise level, so differences of about 1 pp or less (for example B vs C) are not conclusive.
- Only one dropout rate and one 20-epoch schedule were tested, so conclusions about dropout are limited to this setup.
- A small 6-layer VGG-style network, not the full ImageNet VGG-16. No weight decay, no tuning of learning rate, no advanced augmentation (Cutout, MixUp).

### Possible improvements
- Run 3 to 5 seeds per configuration and report mean ± std
- Test dropout rates 0.2 to 0.3 and a longer schedule (40 to 60 epochs)
- Add weight decay and label smoothing; try Cutout or AutoAugment
- Try `ARCH = "vgg11"` to study the effect of depth

### Next in this series
**Day 3:** ResNet-18 on CIFAR-10. Skip connections and why they let networks go deeper.

### References
- Krizhevsky, A., Sutskever, I., Hinton, G. (2012). *ImageNet Classification with Deep Convolutional Neural Networks.* NeurIPS.
- Simonyan, K., Zisserman, A. (2014). *Very Deep Convolutional Networks for Large-Scale Image Recognition.* arXiv:1409.1556.
- Ioffe, S., Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift.* ICML.
- Srivastava, N. et al. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting.* JMLR.